# Модуль 14 — LlamaIndex: агент над вашими данными

Этот ноутбук — практика к лекции «LlamaIndex — когда у тебя много данных» (модуль 14 на сайте курса). В модуле 7 вы собирали RAG руками: чанки, эмбеддинги, cosine-поиск — примерно 60 строк на один источник. Здесь тот же конвейер собирает фреймворк, а в конце появляется то, чего в модуле 7 не было: агент, который сам решает, когда лезть в ваши данные.

Ноутбук идёт по мотивам [Unit 2.2 курса Hugging Face agents-course](https://huggingface.co/learn/agents-course/unit2/llama-index/introduction) (Apache-2.0). Код юнита адаптирован: вместо облачного Inference API с токеном — локальные эмбеддинги и локальная модель, поэтому ядро работает без единого ключа.

Сквозной пример — «картотека персон»: 50 коротких описаний людей из публичного датасета. Реальный двойник — база профилей клиентов в саппорте или CRM: та же лестница «папка выгрузок → индекс → инструмент агента».

**Что вы получите на выходе:**

- конвейер модуля 7, собранный фреймворком: файлы → `Document` → `Node` → Chroma → индекс;
- retriever — поиск по смыслу без LLM и без ключей (главная keyless-вставка модуля);
- query engine — та же база отвечает связным текстом (нужна локальная модель);
- агент с двумя инструментами, который сам решает, когда искать в базе, а когда посчитать файлы, — и стриминг его решений;
- задачи-доработки, включая второй индекс над публичным годовым отчётом (SEC filing) — артефакт ДЗ.

**Карта ноутбука:**

- **Блок 1 (ядро, keyless)** — корпус: 50 персон из датасета, встроенный fallback без сети.
- **Блок 2 (ядро, keyless)** — из файлов в `Document` и `Node`: конвейер загрузки и нарезки.
- **Блок 3 (ядро, keyless)** — Chroma и индекс: эмбеддинги, которые переживают перезапуск.
- **Блок 4 (ядро, keyless)** — retriever: поиск по смыслу без LLM. Гвоздь ноутбука.
- **Блок 5 (keyless-safe)** — автодискавери локальной модели: LM Studio / Ollama.
- **Блок 6 (опционально, нужен локальный сервер)** — query engine: ответ словами.
- **Блок 7 (tool — keyless, агент — нужен сервер)** — данные становятся инструментом.
- **Блок 8** — сводка прогона: что исполнилось, что пропущено.
- **Задачи** — шесть доработок, от своих документов до агента с двумя базами.

Главное про запуск: ядро исполняется **целиком и без единого ключа** — `Run all`, без правок. Интернет нужен в начале: поставить библиотеки, скачать датасет (при недоступности Hub сработает встроенный fallback) и один раз — веса embedding-модели (~130 МБ). Блоки 6–7 без локального сервера модели печатают причину и мягко пропускаются — keyless-прогон остаётся зелёным. Подробности запуска — в [README папки](https://github.com/ITrubnikov/Train_of_Thought-homework/tree/main/notebooks/module-14-llamaindex).

## Подготовка окружения

LlamaIndex ставится по частям. Формула со страницы [LlamaHub](https://huggingface.co/learn/agents-course/unit2/llama-index/llama-hub) юнита: `pip install llama-index-{component-type}-{framework-name}`, а путь import повторяет имя пакета (`llama-index-vector-stores-chroma` → `llama_index.vector_stores.chroma`). Каталог всех интеграций — [llamahub.ai](https://llamahub.ai/). Нам нужно ядро и четыре интеграции:

- `llama-index-core` — `Document`, `Node`, индекс, retriever, query engine, агенты;
- `llama-index-embeddings-huggingface` — локальные embedding-модели: веса скачиваются один раз, считает CPU;
- `llama-index-vector-stores-chroma` — обёртка над векторной базой Chroma;
- `llama-index-llms-openai-like` — класс `OpenAILike` для любого OpenAI-совместимого сервера (LM Studio, Ollama);
- `llama-index-readers-file` — учит `SimpleDirectoryReader` читать PDF и другие форматы (пригодится в Задаче 4).

Отдельно: `chromadb` — сама база; `datasets` — датасет персон с HF Hub; `openai` — клиент, которым `OpenAILike` ходит на сервер (он приедет и транзитивно, но версию пиним явно, чтобы она не уплыла); `nest-asyncio` — страховка для asyncio внутри Jupyter. Версии запинены — прогон воспроизводится одинаково в Colab, Kaggle и локально.

Нужен **Python 3.10+**: в Colab и Kaggle это уже так, локально проверьте `python --version` (системный python3 на macOS — 3.9, не подойдёт; запасной путь — `uv venv --python 3.11`). Ячейка ниже проверит версию сама. Для установки нужен интернет; в Kaggle включите `Notebook options → Internet → On`.

In [ ]:
import sys

assert sys.version_info >= (3, 10), (
    f"нужен Python 3.10+, у вас {sys.version_info.major}.{sys.version_info.minor}. "
    "В Colab/Kaggle это уже так; локально создайте venv поновее, например: uv venv --python 3.11"
)

%pip install -q llama-index-core==0.14.23 llama-index-embeddings-huggingface==0.7.0 llama-index-vector-stores-chroma==0.5.5 llama-index-llms-openai-like==0.7.2 llama-index-readers-file==0.6.0 chromadb==1.5.9 datasets==5.0.0 openai==2.46.0 nest-asyncio==1.6.0 requests

print("[ok] зависимости установлены")

In [ ]:
import importlib.metadata as importlib_metadata
import logging

import nest_asyncio

nest_asyncio.apply()   # вложенные event loop в Jupyter: страховка для sync-вызовов поверх asyncio

# приглушаем безвредный INFO-шум (по одному логгеру, не глобально)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)

RUN_STATUS = {}        # журнал LLM-блоков для сводки в Блоке 8

for pkg in ["llama-index-core", "llama-index-embeddings-huggingface",
            "llama-index-vector-stores-chroma", "llama-index-llms-openai-like",
            "llama-index-readers-file", "chromadb", "datasets", "openai"]:
    print(f"{pkg:44s} {importlib_metadata.version(pkg)}")

print()
print("[ok] окружение готово, Python", sys.version.split()[0])

## Блок 1 (ядро, keyless). Корпус: картотека из 50 персон

Основа всего модуля — «картотека персон»: короткие описания людей из публичного датасета [`dvilasuero/finepersonas-v0.1-tiny`](https://huggingface.co/datasets/dvilasuero/finepersonas-v0.1-tiny) (5000 синтетических персон; `load_dataset` работает без токена). Тот же датасет используют ноутбуки Unit 2.2 — мы сознательно берём его же, чтобы код лекции и практики совпадал. Реальный двойник корпуса — выгрузка профилей клиентов из CRM: такая же папка коротких текстовых карточек.

Берём срез в **50 персон**, а не все 5000: эмбеддинг пяти тысяч файлов — это минуты ожидания, а для поиска по смыслу достаточно пятидесяти. Датасет английский, поэтому и запросы дальше в ноутбуке английские — свои русские документы вы подключите в Задаче 1.

Ячейка устойчива к отсутствию сети: если HF Hub недоступен, срабатывает встроенный fallback-список из 10 персон того же датасета — ядро ноутбука не падает. Ниже вы увидите, сколько файлов записано и как выглядит первый из них.

In [ ]:
import shutil
from pathlib import Path

# Fallback: 10 персон из того же датасета, зашиты в ячейку - ядро не зависит от Hub
FALLBACK_PERSONAS = [
    "A local art historian and museum professional interested in 19th-century American art "
    "and the local cultural heritage of Cincinnati.",
    "A military historian specializing in Japanese and Soviet conflicts during the interwar "
    "period, with an interest in the intricacies of 20th-century geopolitics and the impact "
    "of totalitarian regimes on military strategy.",
    "A chiropractor or physical therapist interested in spinal health and anatomy.",
    "A linguistics student, likely an undergraduate, interested in sociolinguistics and "
    "language study.",
    "A public health analyst specializing in obesity research and weight loss market trends.",
    "A medical doctor or healthcare professional with a focus on educating the general public "
    "about infectious diseases, particularly respiratory infections such as COVID-19, the flu, "
    "and the common cold.",
    "An Egyptologist specializing in ancient Egyptian art and archaeology, likely with a focus "
    "on the Second Intermediate Period and the stylistic evolution of royal iconography.",
    "A gardening expert or horticultural educator with experience in plant care and gardening "
    "practices.",
    "A financial analyst specializing in the development of credit systems and Islamic banking "
    "in emerging economies.",
    "An ornithologist, likely with a focus on penguin behavior and ecology.",
]

try:
    from datasets import load_dataset

    dataset = load_dataset("dvilasuero/finepersonas-v0.1-tiny", split="train")
    personas = [row["persona"] for row in dataset.select(range(50))]
    print(f"[ok] датасет доступен: {len(dataset)} персон, берём срез {len(personas)}")
except Exception as e:
    personas = list(FALLBACK_PERSONAS)
    print(f"[skip] Hub недоступен -> fallback-корпус из {len(personas)} персон ({e!r})")

DATA_DIR = Path("data")
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)   # идемпотентность: каждый прогон собирает корпус заново
DATA_DIR.mkdir()
for i, text in enumerate(personas):
    (DATA_DIR / f"persona_{i:03d}.txt").write_text(text, encoding="utf-8")

files = sorted(DATA_DIR.glob("persona_*.txt"))
print(f"[ok] записано файлов: {len(files)}")
print()
print("Первый файл, persona_000.txt:")
print(files[0].read_text(encoding="utf-8"))

## Блок 2 (ядро, keyless). Из файлов — в Document и Node

В модуле 7 вы делали это руками: резали текст на чанки и считали эмбеддинг каждого чанка. В LlamaIndex оба шага — это два transformations внутри одного конвейера `IngestionPipeline`:

- `SentenceSplitter` режет текст по границам предложений (ваш «чанкер» из модуля 7);
- `HuggingFaceEmbedding` превращает каждый кусок в вектор смысла (ваш `sentence-transformers`).

Два термина фреймворка. `Document` — загруженный файл вместе с метаданными (путь, имя, даты). `Node` — кусок документа, который помнит, откуда он: у каждого node есть текст, эмбеддинг и ссылка на исходный файл. Именно nodes, а не целые файлы, будут находиться при поиске — это тот же принцип «ответ со ссылкой на источник», который вы собирали в модуле 7 вручную.

Embedding-модель — [`BAAI/bge-small-en-v1.5`](https://huggingface.co/BAAI/bge-small-en-v1.5): локальная, ~130 МБ весов скачаются при первом запуске, дальше — кэш. Считает CPU (`device="cpu"`): GPU здесь не нужен, а на Kaggle он и не поможет — корпус крошечный. Загрузчики других форматов и источников — в [docs: loading](https://docs.llamaindex.ai/en/stable/module_guides/loading/connector/), сама страница юнита — [components](https://huggingface.co/learn/agents-course/unit2/llama-index/components).

Ниже вы увидите число documents, число nodes и один node с метаданными.

In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

documents = SimpleDirectoryReader("data").load_data()
print(f"[ok] documents: {len(documents)}")
print("метаданные первого:",
      {k: documents[0].metadata[k] for k in ("file_name", "file_size")})

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5", device="cpu")

pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=512, chunk_overlap=0),
        embed_model,
    ]
)
nodes = await pipeline.arun(documents=documents)

print(f"[ok] nodes: {len(nodes)}, размерность эмбеддинга: {len(nodes[0].embedding)}")
print()
print("Один node:")
print("  текст:    ", nodes[0].text[:90] + "...")
print("  источник: ", nodes[0].metadata["file_name"])

Что значит вывод. 50 документов дали ровно 50 nodes: каждая персона — одно-два предложения, сильно меньше `chunk_size=512`, поэтому нарезка вышла «один файл — один кусок». На длинных документах (как годовой отчёт в Задаче 4) картина другая: один документ рассыпается на десятки nodes. В исходном ноутбуке юнита стоит демо-значение `chunk_size=25` — оно нарочно крошечное, чтобы показать нарезку; в работу его не срисовывайте.

Размерность 384 — это длина вектора смысла у bge-small: каждый кусок текста стал точкой в 384-мерном пространстве, и «похожие по смыслу» куски лежат рядом. А `file_name` в node — та самая ссылка на источник: ответ всегда можно проверить по исходному файлу.

## Блок 3 (ядро, keyless). Chroma: индекс, который переживает перезапуск

Сейчас `nodes` живут в оперативной памяти: закрыли ноутбук — эмбеддинги пропали, при следующем запуске пересчитывать. Для 50 персон это секунды, для реальной базы — минуты и деньги. Лекарство — vector store: `IngestionPipeline` умеет писать эмбеддинги сразу в базу, у нас это [Chroma](https://docs.llamaindex.ai/en/stable/module_guides/storing/vector_stores/) с персистентной папкой `./m14_chroma_db` и коллекцией `personas`.

Поверх заполненного store строится `VectorStoreIndex` — вход во всё остальное: retriever, query engine, агент. Обратите внимание на `embed_model` в обоих местах: **индексация и запросы обязаны использовать одну embedding-модель** — иначе вопрос и куски окажутся в разных векторных пространствах и сравнение станет бессмысленным.

Про идемпотентность: ячейка проверяет `chroma_collection.count()` и наполняет коллекцию только если та пуста — повторный `Run all` не задвоит документы. Обратная сторона: если вы сменили корпус (например, добавили свои файлы), удалите папку `m14_chroma_db` или возьмите новую коллекцию — иначе в индексе останется старый корпус. Реальный двойник: у саппорт-базы индекс живёт в проде постоянно и пополняется инкрементально, а не пересобирается на каждый запрос.

In [ ]:
import chromadb
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore

db = chromadb.PersistentClient(path="./m14_chroma_db")
chroma_collection = db.get_or_create_collection("personas")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

if chroma_collection.count() == 0:
    pipeline = IngestionPipeline(
        transformations=[
            SentenceSplitter(chunk_size=512, chunk_overlap=0),
            embed_model,
        ],
        vector_store=vector_store,   # эмбеддинги едут сразу в Chroma, на диск
    )
    await pipeline.arun(documents=documents)
    print("[ok] корпус проиндексирован в Chroma")
else:
    print("[ok] коллекция уже наполнена -> вставку пропускаем (защита от задвоения)")

print("векторов в коллекции personas:", chroma_collection.count())

index = VectorStoreIndex.from_vector_store(vector_store, embed_model=embed_model)
print("[ok] индекс поверх Chroma готов")

## Блок 4 (ядро, keyless). Retriever: поиск по смыслу без LLM

У индекса три двери. `as_retriever` возвращает список найденных кусков со score. `as_query_engine` — готовый ответ словами. `as_chat_engine` — диалог с памятью. Разница принципиальная: query engine и chat engine **синтезируют** текст, им нужен LLM; retriever — это чистый поиск на эмбеддингах, ему не нужны ни ключ, ни сервер, ни сеть.

Поэтому идём в первую дверь — и это главный keyless-момент модуля: поиск по смыслу работает прямо сейчас, в Colab или Kaggle, без единого ключа. Реальный двойник — подсказка «похожие тикеты» в саппорте: она находит соседей по смыслу, и никакой LLM в ней не участвует.

Два запроса ниже подобраны нарочно. Первый — обычный тематический. Второй — «who can fix my aching back» — не делит **ни одного слова** с карточкой, которую должен найти: если бы это был поиск по словам (grep), он вернул бы пустоту. Смотрите на score и на то, кого нашло.

In [ ]:
retriever = index.as_retriever(similarity_top_k=3)

for query in [
    "someone who studies ancient civilizations",
    "who can fix my aching back",
]:
    print(f"Запрос: {query!r}")
    for result in retriever.retrieve(query):
        print(f"  score={result.score:.3f} | {result.node.metadata['file_name']} | "
              f"{result.node.text[:75]}...")
    print()

print("[ok] поиск по смыслу отработал без LLM и без единого ключа")

Что значит вывод (числа — из реального прогона на срезе 50; на fallback-корпусе из 10 персон score будут немного другими, но лучшие находки — те же).

Первый запрос нашёл египтолога (score 0.582), а следом — математика с бэкграундом в истории и философии математики (0.559) и студента-лингвиста (0.557): карточки со словами «ancient civilizations» в корпусе нет, retriever собрал тех, кто ближе по смыслу. Состав второго-третьего мест у вас может немного отличаться: в текст для эмбеддинга попадают и метаданные файла (включая путь), а он на каждой машине свой.

Второй запрос — главный: «who can fix my aching back» вернул первым «A chiropractor or physical therapist interested in spinal health and anatomy» со score 0.532 — при этом ни одного общего слова между запросом и карточкой. Близки оказались вектора, а не буквы; это и есть разница с grep.

Как читать score: это косинусная близость запроса и куска в пространстве embedding-модели. Сравнивайте score внутри одной выдачи — во втором запросе первое место (0.532) заметно оторвалось от второго (0.432), находка уверенная. Между моделями шкалы не сравниваются: у каждой своя. И отсюда же правило из Блока 3: проиндексировали одной моделью, а спросили другой — пространства разные, score превращаются в шум. Смена embedding-модели всегда означает переиндексацию корпуса.

## Блок 5 (keyless-safe). Локальная модель: автодискавери LM Studio / Ollama

Дальше данным нужен голос: query engine и агент зовут LLM для синтеза ответа и принятия решений. Канон курса прежний — модель на вашей машине из модулей 6.1/6.2: **LM Studio** (порт 1234) или **Ollama** (порт 11434) с instruct-моделью, умеющей tool calling (например, `qwen2.5:7b`). Ячейка ниже сама опросит оба порта и выберет модель; если ваш сервер живёт по другому адресу — задайте переменные окружения `LOCAL_API_BASE` и `LOCAL_MODEL_ID`, они проверяются первыми.

Сервера нет — не страшно: Блоки 6–7 напечатают причину и мягко пропустятся, keyless-прогон останется зелёным. В Colab и Kaggle локального сервера не бывает — там эти блоки пропускаются всегда; живые прогоны делаются локально (вариант C из README).

In [ ]:
import os

import requests

CANDIDATES = [
    ("LM Studio", "http://localhost:1234/v1"),
    ("Ollama", "http://localhost:11434/v1"),
]
if os.environ.get("LOCAL_API_BASE"):
    CANDIDATES.insert(0, ("LOCAL_API_BASE", os.environ["LOCAL_API_BASE"]))

LOCAL_BASE = None    # адрес найденного сервера
LOCAL_MODEL = None   # выбранная модель

for server_name, base in CANDIDATES:
    try:
        r = requests.get(f"{base}/models", timeout=2)
        r.raise_for_status()
        ids = [item["id"] for item in r.json().get("data", [])]
    except Exception:
        continue
    chat_ids = [i for i in ids if "embed" not in i.lower()]   # embedding-модели не годятся
    if chat_ids:
        LOCAL_BASE = base
        LOCAL_MODEL = os.environ.get("LOCAL_MODEL_ID", chat_ids[0])
        print(f"{server_name}: сервер найден на {base} -> модель {LOCAL_MODEL}")
        break

if LOCAL_BASE is None:
    print("Локальный сервер не найден -> живые прогоны Блоков 6-7 пропустятся (для keyless это норма).")

### OpenAILike: один класс для любого OpenAI-совместимого сервера

По формуле пакетов класс для нашего сервера живёт в `llama-index-llms-openai-like` → `llama_index.llms.openai_like`. LM Studio и Ollama говорят по одному протоколу `/v1`, поэтому один `OpenAILike` покрывает оба. Два флага в конструкторе — не украшение: `is_chat_model=True` говорит фреймворку ходить в chat-endpoint, а `is_function_calling_model=True` разрешает агенту отдавать модели инструменты — без него агент Блока 7 молча останется без tools.

Если ваша локальная модель не умеет tool calling, у LlamaIndex есть запасной путь — `ReActAgent`, который работает с любой моделью через текстовый цикл рассуждений; в ядре ноутбука мы держим function calling путь, а про замену — в «Подводных камнях» README.

In [ ]:
llm = None

if LOCAL_BASE is None:
    print("[skip] сервера нет -> llm не создаём, Блоки 6-7 пропустятся.")
else:
    from llama_index.llms.openai_like import OpenAILike

    llm = OpenAILike(
        model=LOCAL_MODEL,
        api_base=LOCAL_BASE,
        api_key="local",                  # заглушка: локальный сервер ключ не проверяет
        is_chat_model=True,               # ходим в /v1/chat/completions, а не в completions
        is_function_calling_model=True,   # без этого флага агент не отдаст модели tools
        context_window=8192,
    )
    try:
        probe = llm.complete("Reply with exactly one word: ready")
        print(f"[ok] {LOCAL_MODEL} на связи, ответ: {str(probe).strip()[:60]}")
    except Exception as e:
        llm = None
        print(f"[fail] сервер найден, но не ответил -> мягкий пропуск: {e!r}")

## Блок 6 (опционально, нужен локальный сервер). Query engine: та же база отвечает словами

Вторая дверь к тому же индексу. Под капотом query engine делает две работы: сначала тот же retrieval, что вы видели в Блоке 4, затем **синтез** — LLM склеивает найденные куски в связный ответ. `response_mode="tree_summarize"` — одна из стратегий синтеза (детальный ответ через дерево сводок).

Вопрос задаём **тот же, что в Блоке 4** — «someone who studies ancient civilizations», — чтобы сравнить выводы двух дверей на одинаковом входе. Подробнее про устройство — [docs: query engine](https://docs.llamaindex.ai/en/stable/module_guides/deploying/query_engine/usage_pattern/).

In [ ]:
if llm is None:
    RUN_STATUS["Блок 6: query engine"] = "skipped"
    print("[skip] Блок 6: локальной модели нет -> query engine не запускаем.")
else:
    try:
        query_engine = index.as_query_engine(llm=llm, response_mode="tree_summarize")
        response = query_engine.query("someone who studies ancient civilizations")
        print("Ответ query engine:")
        print(str(response))
        print()
        print(f"[ok] кусков-источников использовано: {len(response.source_nodes)}")
        RUN_STATUS["Блок 6: query engine"] = "ran"
    except Exception as e:
        RUN_STATUS["Блок 6: query engine"] = "skipped"
        print(f"[fail] живой вызов не прошёл -> мягкий пропуск: {e!r}")

Сравните с Блоком 4. Retriever вернул список кусков со score — сырьё. Query engine на тот же вопрос ответил связным текстом: в нашем прогоне ответ начинался с «An Egyptologist specializing in ancient Egyptian art and archaeology...» и опирался на 2 куска-источника (`source_nodes` — по умолчанию query engine берёт top-2). Формула модуля: **retrieval живёт на эмбеддингах и бесплатен, синтез требует LLM** — потому без сервера у вас работал Блок 4, но не работает этот.

Стратегий синтеза три: `refine` (LLM проходит куски по одному), `compact` (по умолчанию: куски склеиваются, вызовов меньше), `tree_summarize` (дерево сводок). Покрутить их на одном вопросе — Задача 6.

## Блок 7 (tool — keyless, агент — нужен сервер). Данные становятся инструментом

Из модулей 10 и 11.5 вы знаете: tool — это функция с паспортом, а паспорт (имя, описание, схема аргументов) — контракт, который читает модель. В LlamaIndex обёртка называется `FunctionTool`: отдаёте функцию — имя берётся из имени функции, описание из docstring. Подробности — на странице [tools](https://huggingface.co/learn/agents-course/unit2/llama-index/tools) юнита.

Наша функция `count_personas` считает файлы картотеки — скоро она пригодится агенту как «второй инструмент» рядом с базой. Сначала убедимся в главном: `tool.call()` — это обычный вызов Python-функции, никакого LLM в нём нет.

In [ ]:
from llama_index.core.tools import FunctionTool


def count_personas() -> int:
    """Return the number of persona files in the card index (data/ folder)."""
    return len(list(Path("data").glob("persona_*.txt")))


count_tool = FunctionTool.from_defaults(count_personas)

print("Паспорт tool-а, который увидит модель:")
print("  name:       ", count_tool.metadata.name)
print("  description:", count_tool.metadata.description.strip().splitlines()[-1])
print()
print("[ok] tool.call() без всякого LLM:", count_tool.call())

### Агент решает, когда искать

Ключевой ход модуля — три строки, которыми query engine из Блока 6 сам становится инструментом: `QueryEngineTool.from_defaults(query_engine, name=..., description=...)`. Дальше `AgentWorkflow.from_tools_or_functions` собирает агента с двумя инструментами: базой персон и счётчиком файлов.

Чем это отличается от RAG модуля 7? RAG ищет **всегда** — каждый вопрос проходит через индекс. Агент **решает**: лезть в базу, дёрнуть счётчик или ответить сразу. Решение он принимает по `description` инструментов — сломаете описание, и агент перестанет заглядывать в ваши данные (вы сделаете это нарочно в Задаче 3). Реальный двойник: саппорт-бот выбирает между инструментом «база клиентов» и инструментом «калькулятор возврата» — по их описаниям.

Стриминг событий — окно в решения агента: `ToolCallResult` показывает, какой tool вызван, с какими аргументами и что вернул, `AgentStream` — дельты финального текста. Два вопроса ниже подобраны как пара: первый должен отправить агента в базу (`personas`), второй — к счётчику (`count_personas`). Агенты LlamaIndex асинхронны, отсюда `async for` и `await` — в Jupyter это работает прямо в ячейке ([docs: async python](https://docs.llamaindex.ai/en/stable/getting_started/async_python/); сам паттерн — [docs: AgentWorkflow](https://docs.llamaindex.ai/en/stable/examples/agent/agent_workflow_basic/), страница юнита — [agents](https://huggingface.co/learn/agents-course/unit2/llama-index/agents)).

In [ ]:
if llm is None:
    RUN_STATUS["Блок 7: агент"] = "skipped"
    print("[skip] Блок 7: локальной модели нет -> агента не запускаем.")
else:
    try:
        from llama_index.core.agent.workflow import (
            AgentStream,
            AgentWorkflow,
            ToolCallResult,
        )
        from llama_index.core.tools import QueryEngineTool

        query_engine_tool = QueryEngineTool.from_defaults(
            query_engine=index.as_query_engine(llm=llm, similarity_top_k=3),
            name="personas",
            description=("Search the card index of personas: find people "
                         "by interests, profession or topic."),
        )
        agent = AgentWorkflow.from_tools_or_functions(
            [query_engine_tool, count_tool],
            llm=llm,
            system_prompt=("You are a helpful assistant with access to a card index "
                           "of persona descriptions."),
        )

        for question in [
            "Who in the card index studies ancient civilizations? Answer briefly.",
            "How many personas are in the card index?",
        ]:
            print("=" * 72)
            print("Вопрос:", question)
            handler = agent.run(question)
            async for ev in handler.stream_events():
                if isinstance(ev, ToolCallResult):
                    print(f"\n[tool] {ev.tool_name}({ev.tool_kwargs}) -> "
                          f"{str(ev.tool_output)[:110]}...")
                elif isinstance(ev, AgentStream):
                    print(ev.delta, end="", flush=True)
            await handler
            print()
        RUN_STATUS["Блок 7: агент"] = "ran"
    except Exception as e:
        RUN_STATUS["Блок 7: агент"] = "skipped"
        print(f"[fail] живой прогон не прошёл -> мягкий пропуск: {e!r}")

### Память: Context

По умолчанию агент stateless: каждый `run` — с чистого листа, прошлый вопрос забыт. Память в LlamaIndex — не магия, а объект, который вы передаёте сами: `ctx = Context(agent)`, затем `agent.run(..., ctx=ctx)` в каждом вызове. В нашем прогоне на второй вопрос агент ответил «You mentioned your name is Bob.» — состояние доехало через `ctx`, не через модель.

In [ ]:
if llm is None:
    RUN_STATUS["Блок 7: память (Context)"] = "skipped"
    print("[skip] память не проверяем: локальной модели нет.")
else:
    try:
        from llama_index.core.workflow import Context

        ctx = Context(agent)
        r1 = await agent.run("My name is Bob.", ctx=ctx)
        r2 = await agent.run("What was my name again?", ctx=ctx)
        print("Первый ответ: ", str(r1)[:160])
        print("Второй ответ: ", str(r2)[:160])
        RUN_STATUS["Блок 7: память (Context)"] = "ran"
    except Exception as e:
        RUN_STATUS["Блок 7: память (Context)"] = "skipped"
        print(f"[fail] живой прогон не прошёл -> мягкий пропуск: {e!r}")

## Блок 8. Сводка прогона

Самопроверка перед сдачей: ядро обязано быть `ran` целиком; LLM-блоки — `ran` при локальном сервере или `skipped` в Colab/Kaggle, и то и другое — валидный прогон.

In [ ]:
core_blocks = {
    "Блок 1: корпус": f"ran ({len(files)} файлов)",
    "Блок 2: Document -> Node": f"ran ({len(nodes)} nodes)",
    "Блок 3: Chroma + индекс": f"ran ({chroma_collection.count()} векторов)",
    "Блок 4: retriever без LLM": "ran",
    "Блок 7: FunctionTool без LLM": f"ran (count_personas() = {count_tool.call()})",
}
llm_blocks = {
    name: RUN_STATUS.get(name, "skipped")
    for name in ["Блок 6: query engine", "Блок 7: агент", "Блок 7: память (Context)"]
}

print("Ядро (keyless):")
for name, status in core_blocks.items():
    print(f"  {name:34s} {status}")
print()
print("LLM-блоки (нужен локальный сервер):")
for name, status in llm_blocks.items():
    print(f"  {name:34s} {status}")
print()
if all(status == "ran" for status in llm_blocks.values()):
    print("[ok] полный прогон: ядро + живая модель.")
else:
    print("[ok] keyless-прогон зелёный; блоки со skipped запускаются локально "
          "(вариант C из README).")

## Задачи — доработайте рабочий код

Правило прежнее: каждая задача опирается на рабочий образец из ноутбука — вы копируете ячейку-образец и меняете её под себя. Пометки: **(keyless)** — работает без сервера модели; **(нужен сервер)** — потребуется LM Studio/Ollama из Блока 5. Обязательная для ДЗ — retriever-часть Задачи 4; остальные — по желанию, но Задачи 1 и 3 дают больше всего понимания на строчку кода.

### Задача 1 (keyless). Свои документы

Образец — Блоки 1–4. Соберите 5 и больше собственных `.txt`/`.md` файлов — конспекты этого курса, README ваших проектов, заметки — и проиндексируйте их вместо персон. Практический смысл: это ровно та лестница, по которой вики компании становится базой знаний бота.

Что учесть:

- складывайте файлы в отдельную папку (например, `my_data/`) — Блок 1 пересоздаёт `data/` при каждом прогоне;
- для русских текстов `bge-small-en` не годится — возьмите многоязычную [`intfloat/multilingual-e5-small`](https://huggingface.co/intfloat/multilingual-e5-small) (~470 МБ весов против ~130 МБ у bge-small — качается дольше, но остаётся локальной и CPU-совместимой). У семейства e5 запрос и документ принято помечать префиксами — `HuggingFaceEmbedding` передаёт их параметрами: `query_instruction="query: "`, `text_instruction="passage: "`;
- индекс собирайте в новой коллекции той же Chroma-базы (например, `my_docs`) — правило одного векторного пространства: в одной коллекции не смешивают эмбеддинги разных моделей.

Критерий приёмки: retriever находит нужный документ по смысловому запросу **без точного совпадения слов** — в ноутбуке показан вывод top-3 со score (как в Блоке 4).

### Задача 2 (keyless). Метаданные и фильтры

Образец — Блоки 2–4. Поиск по смыслу дружит с обычным WHERE: у каждого `Document` есть словарь `metadata`, он доезжает до nodes и до Chroma, и retriever умеет фильтровать по нему до поиска.

Проставьте документам категории при создании (например, `Document(text=..., metadata={"category": "health"})` — или пройдитесь по `documents` из Блока 2 и заполните `doc.metadata["category"]` по содержимому). Затем сравните выдачу с фильтром и без:

```python
from llama_index.core.vector_stores import ExactMatchFilter, MetadataFilters

filters = MetadataFilters(filters=[ExactMatchFilter(key="category", value="health")])
retriever_filtered = index.as_retriever(similarity_top_k=3, filters=filters)
```

(Имена классов — из запиненной `llama-index-core==0.14.23`.) Не забудьте пересобрать индекс в новой коллекции: у уже записанных в Chroma nodes метаданные задним числом не поменяются.

Критерий приёмки: один и тот же запрос с фильтром и без даёт разные top-3, оба вывода показаны в ноутбуке.

### Задача 3 (нужен сервер). Сломайте description

Образец — агент-ячейка Блока 7. Скопируйте её, замените `description` у `personas`-tool на бесполезное «useful tool» — и повторите первый вопрос (про древние цивилизации).

Критерий приёмки: в стриминге `ToolCallResult` видно, что выбор инструмента изменился или сломался — агент не зовёт `personas`, зовёт не то или отвечает без поиска. Плюс один абзац вашими словами: почему это произошло. Мост к модулю 11.5: description — контракт, который читает модель; агент выбирает инструменты только по паспортам, кода он не видит.

### Задача 4 (retriever — keyless, агент — нужен сервер). Второй индекс: годовой отчёт

**Артефакт ДЗ по roadmap курса.** Реальный двойник здесь без кавычек — публичная финансовая отчётность: годовой отчёт 10-K с [SEC EDGAR](https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany). Retriever-часть обязательна, агентная — при локальном сервере.

Шаг 1 — скачать и почистить отчёт (образец кода; sec.gov без заголовка `User-Agent` с контактом отдаёт 403 — это их правило для роботов, заголовок уже прописан):

```python
from pathlib import Path

import requests
from bs4 import BeautifulSoup   # beautifulsoup4 приехал вместе с llama-index-readers-file

# 10-K Apple за 2025 финансовый год (~1,5 МБ HTML)
URL = ("https://www.sec.gov/Archives/edgar/data/320193/"
       "000032019325000079/aapl-20250927.htm")
HEADERS = {"User-Agent": "training-notebook your-email@example.com"}

html = requests.get(URL, headers=HEADERS, timeout=30).text
text = BeautifulSoup(html, "html.parser").get_text(" ", strip=True)
Path("filings").mkdir(exist_ok=True)
Path("filings/aapl-10k-2025.txt").write_text(text, encoding="utf-8")
print(len(html), "символов HTML ->", len(text), "символов текста")
```

Запасной путь: любой публичный годовой отчёт в PDF — просто положите файл в `filings/`, `SimpleDirectoryReader` прочитает PDF из коробки (`llama-index-readers-file` уже установлен).

Шаг 2 — по образцу Блоков 2–4: `SimpleDirectoryReader("filings")`, тот же pipeline, **вторая коллекция `"filings"`** в той же Chroma-базе, второй retriever. Для масштаба: в нашем прогоне из этого 10-K получилось ~220 тысяч знаков текста и 121 node — вот где `SentenceSplitter` работает по-настоящему.

Шаг 3 (при сервере) — по образцу Блока 7: второй `QueryEngineTool` с `name="filings"` и честным description, агент с двумя базами.

Критерии приёмки (это и есть чек-лист ДЗ из README):

- retriever по отчёту отвечает на фактический вопрос — например, «What was the total net sales in fiscal 2025?» или вопрос про выручку конкретного сегмента; top-3 со score показаны;
- (при сервере) в стриминге `ToolCallResult` видно: на вопрос про финансы агент выбрал `filings`, на вопрос про людей — `personas`.

### Задача 5 (нужен сервер). Оценка качества: FaithfulnessEvaluator

Образец — Блок 6. «Ответил» не значит «ответил по документам»: query engine может дописать от себя. Встроенный [`FaithfulnessEvaluator`](https://docs.llamaindex.ai/en/stable/module_guides/evaluating/) проверяет LLM-ом, подтверждён ли ответ найденными кусками:

```python
from llama_index.core.evaluation import FaithfulnessEvaluator

evaluator = FaithfulnessEvaluator(llm=llm)
response = query_engine.query("...")
print(evaluator.evaluate_response(response=response).passing)
```

Прогоните 5 вопросов к базе персон, из них один заведомо «не из документов» (например, про погоду на Марсе).

Критерий приёмки: таблица «вопрос → passing» в выводе или markdown; провокационный вопрос помечен; одним предложением — вывод: галлюцинирует ли ваш query engine, когда в базе нет ответа.

### Задача 6 (частично keyless). Тюнинг retrieval

Качество RAG — это ручки, а не магия. Покрутите три из них на одном и том же вопросе:

- `similarity_top_k` у retriever — 1 / 3 / 10 (keyless, образец — Блок 4);
- `chunk_size` у `SentenceSplitter` — например, 128 против 512 на корпусе `filings` из Задачи 4 (keyless; пересоберите индекс в отдельной коллекции, чтобы не смешивать нарезки);
- `response_mode` у query engine — `compact` / `tree_summarize` / `refine` (нужен сервер, образец — Блок 6).

Критерий приёмки: markdown-таблица сравнения — вариант, ответ (или top-3), время — и вывод одним-двумя предложениями: какая ручка на что повлияла.

## Что дальше

Сдача ДЗ — как обычно: прогнанный ноутбук (ядро целиком, сводка Блока 8 зелёная) плюс Задача 4, публичная ссылка в чат курса в формате `[Модуль 14, ДЗ] {ссылка}`. Полный чек-лист критериев — в README папки.

За кадром ноутбука осталась вторая опора LlamaIndex — workflows: событийные конвейеры со степами и type-аннотациями, где процесс идёт по вашим рельсам, а не по решениям агента. Они разобраны в секции лекции и вернутся всерьёз в модуле 15 (LangGraph); практики по ним в этом ДЗ нет — это сознательное решение, не пропуск ([обзор в docs](https://docs.llamaindex.ai/en/stable/understanding/workflows/), [страница юнита](https://huggingface.co/learn/agents-course/unit2/llama-index/workflows)).

Куда смотреть дальше:

- страницы Unit 2.2, по которым шёл ноутбук: [introduction](https://huggingface.co/learn/agents-course/unit2/llama-index/introduction), [llama-hub](https://huggingface.co/learn/agents-course/unit2/llama-index/llama-hub), [components](https://huggingface.co/learn/agents-course/unit2/llama-index/components), [tools](https://huggingface.co/learn/agents-course/unit2/llama-index/tools), [agents](https://huggingface.co/learn/agents-course/unit2/llama-index/agents), [workflows](https://huggingface.co/learn/agents-course/unit2/llama-index/workflows);
- документация по темам блоков: [RAG](https://docs.llamaindex.ai/en/stable/understanding/rag/), [загрузчики](https://docs.llamaindex.ai/en/stable/module_guides/loading/connector/), [vector stores](https://docs.llamaindex.ai/en/stable/module_guides/storing/vector_stores/), [query engine](https://docs.llamaindex.ai/en/stable/module_guides/deploying/query_engine/usage_pattern/), [агенты](https://docs.llamaindex.ai/en/stable/understanding/agent/), [оценка качества](https://docs.llamaindex.ai/en/stable/module_guides/evaluating/);
- каталог интеграций [LlamaHub](https://llamahub.ai/): loaders, vector stores, tools — всё по одной формуле пакетов.

Дальше по курсу — модуль 12 (OpenClaw), а LlamaIndex остаётся вторым из трёх фреймворков курса: smolagents из модуля 10 крутится вокруг кода агента, LlamaIndex — вокруг данных, LangGraph в модуле 15 — вокруг графа процесса.

Ноутбук идёт по мотивам Unit 2.2 курса Hugging Face agents-course (Apache-2.0).